# 05주차 · 문장·문서 임베딩과 의미 유사도

**임베딩 기반 데이터 과학 — 국립목포대학교 컴퓨터학부 4학년**

이 Notebook은 *Hands-On Large Language Models* 공개 저장소의 개념과 실습 흐름을
한국어 수업에 맞게 새로 구성한 파생 강의자료입니다. 원본은 Apache License 2.0을
따르며, 출처와 변경 사항은 `SOURCE_AND_LICENSE.md`에 기록했습니다.

- 원본: https://github.com/HandsOnLLM/Hands-On-Large-Language-Models
- 기준 커밋: `ea3390819997999a51983677b80b3aac4dc50ada`
- 권장 환경: Google Colab 또는 Python 3.11+


## 학습목표

- 문장 임베딩과 평균 토큰 임베딩을 구분한다.
- Bi-인코더의 검색 효율을 설명한다.
- 대조학습의 양성·비관련 쌍을 설계한다.


In [ ]:
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 100)

def cosine(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(a @ b / denom) if denom else 0.0


## Pooling과 패딩 마스크

토큰 은닉 상태를 평균해 문장 벡터를 만들 때 패딩을 제외해야 한다. 이 계산은 4주차의 마스크 개념을 문장 임베딩으로 연결한다.


In [ ]:
token_states = np.array([[1.0, 0.0], [0.8, 0.2], [0.0, 0.0]])
attention_mask = np.array([1, 1, 0], dtype=float)
mean_with_padding = token_states.mean(axis=0)
masked_mean = (token_states * attention_mask[:, None]).sum(axis=0) / attention_mask.sum()
pd.DataFrame([mean_with_padding, masked_mean], index=["padding 포함", "padding 제외"], columns=["dim1", "dim2"]).round(3)


In [ ]:
sentences = [
    "청년 창업 자금을 지원합니다",
    "청년 사업가에게 사업비를 제공합니다",
    "고령자 이동 복지 서비스를 확대합니다",
    "섬 지역 원격진료를 지원합니다",
]
query = "청년의 창업 비용을 도와주는 정책"


In [ ]:
# 항상 실행되는 희소 기준선
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
vec = TfidfVectorizer(analyzer="char", ngram_range=(2, 4))
X = vec.fit_transform([query, *sentences])
baseline_scores = cosine_similarity(X[0], X[1:]).ravel()
pd.DataFrame({"문장": sentences, "TF-IDF 점수": baseline_scores}).sort_values("TF-IDF 점수", ascending=False)


In [ ]:
# 주차별 연습에서는 선택 셀이다. 과제 1·2에서는 실행하거나 교수자가 제공한 저장 임베딩을 사용한다.
RUN_SENTENCE_MODEL = False  # 인터넷 가능한 Colab에서 True
if RUN_SENTENCE_MODEL:
    from sentence_transformers import SentenceTransformer
    model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    model = SentenceTransformer(model_name)
    E = model.encode([query, *sentences], normalize_embeddings=True)
    dense_scores = E[1:] @ E[0]
    comparison = pd.DataFrame({"문장": sentences, "TF-IDF 점수": baseline_scores, "밀집 점수": dense_scores})
    print(comparison.sort_values("밀집 점수", ascending=False).to_string(index=False))
else:
    print("문장 임베딩 선택 실습을 건너뜁니다. 과제에서는 실행 또는 저장 임베딩이 필요합니다.")


## 대조학습 설계 활동

다음 형식으로 최소 10개의 한국어 학습 쌍을 작성하라.

|앵커|양성 문장|혼동하기 쉬운 비관련 문장|
|---|---|---|
|청년 창업 지원|청년 사업비 제공|청년 주거비 지원|

음성 문장이 너무 쉬우면 모델이 세밀한 의미 차이를 배우지 못한다.


---
## 학습 기록과 생성형 AI 사용 내역

다음 항목을 자신의 말로 작성하세요.

1. 이번 실습에서 가장 중요한 결과는 무엇인가?
2. 결과를 뒷받침하는 수치 또는 그래프는 무엇인가?
3. 실패하거나 예상과 달랐던 부분은 무엇인가?
4. 생성형 AI를 사용했다면 프롬프트, 채택·거부한 제안, 직접 검증한 내용을 기록하라.
